# Notebook 07: Pub/Sub — Real-Time Messaging with Redis

**Pub/Sub** (Publish/Subscribe) is a messaging pattern where:
- **Publishers** send messages to **channels**
- **Subscribers** listen on channels and receive messages in real-time

Think of it like **radio broadcasting**:
- A radio station (publisher) broadcasts on a frequency (channel)
- Anyone tuned to that frequency (subscriber) hears the broadcast
- If nobody is listening, the broadcast is lost

```
Publisher A ─────► [channel: news]  ─────► Subscriber 1
                                    ─────► Subscriber 2
Publisher B ─────► [channel: sports] ────► Subscriber 2
                                    ────► Subscriber 3
```

**Key trait:** Fire-and-forget. If no one is listening, the message is **lost forever**.

In [ ]:
import redis
import threading
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
print("Connected!")

# Note: Pub/Sub requires separate connections for publish and subscribe
# The subscribe connection is "dedicated" — it can only receive messages

---
## 1. Basic PUBLISH — Sending Messages

In [ ]:
# PUBLISH sends a message to a channel and returns how many subscribers received it
# Redis CLI: PUBLISH news "Hello World"

listeners = r.publish('news', 'Hello World!')
print(f"Subscribers who received the message: {listeners}")
print("0 because nobody is listening yet!")

---
## 2. Basic SUBSCRIBE — Receiving Messages

Subscribe is **blocking** — it waits forever for messages. In a notebook, we use **threads** to demonstrate.

In [ ]:
# Create a separate connection for subscribing
# (a subscribed connection can ONLY receive messages)

received_messages = []

def subscriber_worker(channel, max_messages=3):
    """Subscribe to a channel and collect messages."""
    sub_conn = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
    pubsub = sub_conn.pubsub()
    pubsub.subscribe(channel)
    
    count = 0
    for message in pubsub.listen():
        # Messages have a type: 'subscribe', 'message', 'unsubscribe'
        if message['type'] == 'message':
            received_messages.append(message)
            print(f"  [Subscriber] Got: '{message['data']}' on channel '{message['channel']}'")
            count += 1
            if count >= max_messages:
                break
    
    pubsub.unsubscribe()
    pubsub.close()

# Start subscriber in a background thread
t = threading.Thread(target=subscriber_worker, args=('news', 3), daemon=True)
t.start()
time.sleep(0.5)  # Give subscriber time to connect

# Now publish some messages
for i, msg in enumerate(['Breaking news!', 'Weather update', 'Sports scores'], 1):
    listeners = r.publish('news', msg)
    print(f"  [Publisher]  Sent: '{msg}' → {listeners} listener(s)")
    time.sleep(0.2)

t.join(timeout=3)
print(f"\nTotal messages received: {len(received_messages)}")

---
## 3. Message Format

Every message from `pubsub.listen()` is a dictionary:

In [ ]:
# Let's look at the raw message format
if received_messages:
    print("Raw message structure:")
    for msg in received_messages:
        print(f"  {msg}")

print("\nMessage fields:")
print("  type    — 'subscribe', 'message', or 'unsubscribe'")
print("  channel — which channel the message came from")
print("  data    — the actual message content")
print("  pattern — None for direct subscribe, pattern string for PSUBSCRIBE")

---
## 4. Multiple Channels

In [ ]:
multi_messages = []

def multi_channel_subscriber():
    sub_conn = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
    pubsub = sub_conn.pubsub()
    # Subscribe to multiple channels at once
    pubsub.subscribe('news', 'sports', 'tech')
    
    count = 0
    for message in pubsub.listen():
        if message['type'] == 'message':
            multi_messages.append(message)
            print(f"  [{message['channel']}] {message['data']}")
            count += 1
            if count >= 4:
                break
    pubsub.close()

t = threading.Thread(target=multi_channel_subscriber, daemon=True)
t.start()
time.sleep(0.5)

# Publish to different channels
r.publish('news', 'Election results are in!')
r.publish('sports', 'India wins the match!')
r.publish('tech', 'New Python version released')
r.publish('news', 'Market update')

t.join(timeout=3)
print(f"\nReceived from {len(set(m['channel'] for m in multi_messages))} different channels")

---
## 5. Pattern Subscribe (PSUBSCRIBE)

Subscribe to all channels matching a **glob pattern**. This is incredibly powerful!

In [ ]:
pattern_messages = []

def pattern_subscriber():
    sub_conn = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
    pubsub = sub_conn.pubsub()
    # Subscribe to ALL channels starting with "events:"
    pubsub.psubscribe('events:*')
    
    count = 0
    for message in pubsub.listen():
        if message['type'] == 'pmessage':  # Note: 'pmessage' for pattern matches!
            pattern_messages.append(message)
            print(f"  [pattern={message['pattern']}] channel={message['channel']}: {message['data']}")
            count += 1
            if count >= 4:
                break
    pubsub.close()

t = threading.Thread(target=pattern_subscriber, daemon=True)
t.start()
time.sleep(0.5)

# These ALL match the 'events:*' pattern
r.publish('events:user:signup', '{"user": "alice"}')
r.publish('events:order:placed', '{"order": 42}')
r.publish('events:payment:received', '{"amount": 99.99}')
r.publish('events:user:logout', '{"user": "bob"}')

# This does NOT match
r.publish('other:channel', 'This will not be received')

t.join(timeout=3)
print(f"\nCaught {len(pattern_messages)} events with 'events:*' pattern")

---
## 6. Non-Blocking: get_message()

Instead of the blocking `listen()`, you can **poll** for messages.

In [ ]:
# Create a pubsub object and subscribe
pubsub = r.pubsub()
pubsub.subscribe('updates')

# get_message() returns None if no message is available (non-blocking!)
msg = pubsub.get_message()  # This is the 'subscribe' confirmation
print(f"Subscribe confirmation: {msg}")

msg = pubsub.get_message()  # No message yet
print(f"No message available: {msg}")

# Publish something
r.publish('updates', 'Here is an update!')
time.sleep(0.1)  # Brief pause for message to arrive

msg = pubsub.get_message()  # Now we get it!
print(f"Got message: {msg}")

pubsub.unsubscribe()
pubsub.close()

---
## 7. Message Handlers (Callbacks)

Register callback functions that are automatically called when messages arrive.

In [ ]:
# Define handler functions
handler_log = []

def handle_alerts(message):
    handler_log.append(('alert', message['data']))
    print(f"  ALERT HANDLER: {message['data']}")

def handle_info(message):
    handler_log.append(('info', message['data']))
    print(f"  INFO HANDLER: {message['data']}")

# Subscribe with handlers — the key is the channel name, value is the handler
pubsub = r.pubsub()
pubsub.subscribe(**{
    'alerts': handle_alerts,
    'info': handle_info
})

# Publish messages
r.publish('alerts', 'Server CPU at 95%!')
r.publish('info', 'Deployment complete')
r.publish('alerts', 'Disk space low!')

# Process messages (handlers are called inside get_message)
time.sleep(0.2)
for _ in range(10):  # Process pending messages
    pubsub.get_message()

print(f"\nHandlers processed {len(handler_log)} messages")
pubsub.unsubscribe()
pubsub.close()

---
## 8. Run in Thread (Built-in)

Redis-py has a built-in way to run the subscriber in a background thread.

In [ ]:
thread_messages = []

def my_handler(message):
    thread_messages.append(message['data'])

pubsub = r.pubsub()
pubsub.subscribe(**{'my_channel': my_handler})

# Start a background thread that automatically processes messages
thread = pubsub.run_in_thread(sleep_time=0.01)  # Checks every 10ms

# Publish some messages
for i in range(5):
    r.publish('my_channel', f'Message {i}')

time.sleep(0.5)  # Let messages process

# Stop the thread
thread.stop()

print(f"Received {len(thread_messages)} messages via background thread:")
for msg in thread_messages:
    print(f"  {msg}")

---
## 9. Limitations of Pub/Sub

| Limitation | What it means |
|---|---|
| **No persistence** | If no one is subscribed, the message is lost forever |
| **No acknowledgment** | You don't know if a subscriber actually processed it |
| **No replay** | New subscribers can't see past messages |
| **No consumer groups** | Can't distribute work across multiple consumers |
| **All-or-nothing** | Every subscriber gets every message (can't divide work) |

If you need any of these features, use **Redis Streams** (Notebook 11)!

---
## 10. Real-World: Simple Chat Room

In [ ]:
import json

chat_log = []

def chat_listener(room):
    """Listen for messages in a chat room."""
    sub = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
    ps = sub.pubsub()
    ps.subscribe(f'chat:{room}')
    
    count = 0
    for msg in ps.listen():
        if msg['type'] == 'message':
            data = json.loads(msg['data'])
            chat_log.append(data)
            print(f"  [{data['room']}] {data['user']}: {data['text']}")
            count += 1
            if count >= 4:
                break
    ps.close()

def send_message(room, user, text):
    """Send a message to a chat room."""
    message = json.dumps({'room': room, 'user': user, 'text': text, 'time': time.time()})
    r.publish(f'chat:{room}', message)

# Start listener for 'general' room
t = threading.Thread(target=chat_listener, args=('general',), daemon=True)
t.start()
time.sleep(0.5)

# Users send messages
send_message('general', 'Sujit', 'Hey everyone!')
send_message('general', 'Alice', 'Hi Sujit! Welcome!')
send_message('general', 'Bob', 'What are we learning today?')
send_message('general', 'Sujit', 'Redis Pub/Sub!')

t.join(timeout=3)
print(f"\nChat log has {len(chat_log)} messages")

---
## 11. Real-World: Event Notification System

In [ ]:
event_log = []

def event_listener():
    """Listen for all application events."""
    sub = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
    ps = sub.pubsub()
    ps.psubscribe('app:events:*')  # All events
    
    count = 0
    for msg in ps.listen():
        if msg['type'] == 'pmessage':
            event_type = msg['channel'].replace('app:events:', '')
            event_log.append({'type': event_type, 'data': msg['data']})
            print(f"  EVENT [{event_type}]: {msg['data']}")
            count += 1
            if count >= 4:
                break
    ps.close()

# Start event listener
t = threading.Thread(target=event_listener, daemon=True)
t.start()
time.sleep(0.5)

# Different parts of the app emit events
r.publish('app:events:user.signup', json.dumps({'user_id': 1001, 'name': 'Sujit'}))
r.publish('app:events:order.placed', json.dumps({'order_id': 42, 'total': 99.99}))
r.publish('app:events:payment.received', json.dumps({'order_id': 42, 'method': 'card'}))
r.publish('app:events:email.sent', json.dumps({'to': 'sujit@example.com', 'template': 'welcome'}))

t.join(timeout=3)
print(f"\nCaptured {len(event_log)} events")

---
## Pub/Sub vs Streams vs Lists — When to Use What?

| Feature | Pub/Sub | Lists (Queue) | Streams |
|---|---|---|---|
| Persistence | No | Yes | Yes |
| Multiple consumers | All get same msg | One gets each msg | Consumer groups |
| Replay history | No | No (once popped, gone) | Yes |
| Acknowledgment | No | No | Yes (XACK) |
| Blocking wait | Yes | Yes (BLPOP) | Yes (XREAD BLOCK) |
| Best for | Real-time broadcast | Simple task queue | Robust event streaming |

---
## Key Takeaways

```
PUBLISH channel message         → Send a message
SUBSCRIBE channel1 channel2     → Listen on channels
PSUBSCRIBE pattern*             → Listen on pattern-matched channels
UNSUBSCRIBE channel             → Stop listening
```

### Python API
```python
pubsub = r.pubsub()
pubsub.subscribe('channel')              # Basic subscribe
pubsub.psubscribe('pattern:*')           # Pattern subscribe
pubsub.subscribe(**{'ch': handler_fn})   # With callback
pubsub.listen()                          # Blocking iterator
pubsub.get_message()                     # Non-blocking poll
pubsub.run_in_thread(sleep_time=0.01)    # Background thread
```

---
## Exercises

1. **Multi-Room Chat:** Extend the chat room example to support 3 rooms. Create one subscriber per room. Publish messages to different rooms and verify isolation.

2. **Notification Types:** Create a pattern subscriber for `notify:*`. Publish to `notify:email`, `notify:sms`, `notify:push`. Count how many of each type were received.

3. **Pub/Sub Stats:** Write a monitoring tool that subscribes to all channels (`*`) and prints a real-time count of messages per channel per second.

4. **Cache Invalidation:** When a user profile is updated (simulated), publish an invalidation message. Have a "cache service" subscriber that listens and clears the relevant cache key.

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 08 — Transactions & Pipelines](./08_Transactions_and_Pipelines.ipynb)** — Atomic operations, batching, and safe concurrent updates!